# Download BlueConfig files from BBP simulation campaign

Recursively finds and downloads all `BlueConfig` files from
`s3://openbluebrain/Simulation_data/Simulation_Campaigns/Rat/SSCx/ab2b5cfccf/`,
preserving the original directory nesting structure.

In [1]:
import subprocess
import os
from pathlib import Path

In [ ]:
# S3_ROOT = "s3://openbluebrain/Simulation_data/Simulation_Campaigns/Rat/SSCx/ab2b5cfccf/"
# OUTPUT_ROOT = "/Users/james/Documents/obi/Data/Simulations/BBP-raw/ab2b5cfccf_blueconfigs"

S3_ROOT = "s3://openbluebrain/Simulation_data/Simulation_Campaigns/Rat/SSCx/14f99e7d0e/"
OUTPUT_ROOT = "/Users/james/Documents/obi/Data/Simulations/BBP-raw/14f99e7d0e_blueconfigs"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"S3 root:    {S3_ROOT}")
print(f"Output dir: {OUTPUT_ROOT}")

S3 root:    s3://openbluebrain/Simulation_data/Simulation_Campaigns/Rat/SSCx/14f99e7d0ef/
Output dir: /Users/james/Documents/obi/Data/Simulations/BBP-raw/14f99e7d0e_blueconfigs


## 1. List top-level subdirectories

In [3]:
result = subprocess.run(
    ["aws", "s3", "ls", S3_ROOT, "--no-sign-request"],
    capture_output=True, text=True, check=True,
)

top_level_dirs = []
for line in result.stdout.strip().splitlines():
    line = line.strip()
    if line.startswith("PRE "):
        dirname = line.replace("PRE ", "").strip("/")
        top_level_dirs.append(dirname)

print(f"Found {len(top_level_dirs)} top-level directories:")
for d in top_level_dirs:
    print(f"  {d}")

CalledProcessError: Command '['aws', 's3', 'ls', 's3://openbluebrain/Simulation_data/Simulation_Campaigns/Rat/SSCx/14f99e7d0ef/', '--no-sign-request']' returned non-zero exit status 1.

## 2. Recursively find all BlueConfig files

Uses `aws s3 ls --recursive` on each top-level directory to find all `BlueConfig` files.

In [ ]:
blueconfig_keys = []

for i, dirname in enumerate(top_level_dirs):
    s3_dir = f"{S3_ROOT}{dirname}/"
    print(f"[{i+1}/{len(top_level_dirs)}] Scanning {dirname}/ ...", end=" ")

    result = subprocess.run(
        ["aws", "s3", "ls", s3_dir, "--recursive", "--no-sign-request"],
        capture_output=True, text=True, check=True,
    )

    count = 0
    for line in result.stdout.strip().splitlines():
        # Lines look like: "2024-11-26 20:34:08       6860 Simulation_data/.../BlueConfig"
        parts = line.strip().split(maxsplit=3)
        if len(parts) == 4:
            key = parts[3]
            if key.endswith("BlueConfig"):
                blueconfig_keys.append(key)
                count += 1

    print(f"found {count} BlueConfig file(s)")

print(f"\nTotal BlueConfig files found: {len(blueconfig_keys)}")

In [ ]:
# Preview a few paths
for key in blueconfig_keys[:5]:
    print(key)
if len(blueconfig_keys) > 5:
    print(f"  ... and {len(blueconfig_keys) - 5} more")

## 3. Download all BlueConfig files

Downloads each BlueConfig file individually, preserving the nesting structure relative to `ab2b5cfccf/`.

In [ ]:
# The keys from `aws s3 ls --recursive` are relative to the bucket root, e.g.:
#   Simulation_data/Simulation_Campaigns/Rat/SSCx/ab2b5cfccf/<uuid>/<num>/BlueConfig
# We want to preserve structure relative to ab2b5cfccf/

S3_BUCKET = "s3://openbluebrain/"
PREFIX = "Simulation_data/Simulation_Campaigns/Rat/SSCx/ab2b5cfccf/"

failed = []
for i, key in enumerate(blueconfig_keys):
    # Relative path within ab2b5cfccf/
    rel_path = key[len(PREFIX):]
    local_path = os.path.join(OUTPUT_ROOT, rel_path)
    s3_path = f"{S3_BUCKET}{key}"

    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    print(f"[{i+1}/{len(blueconfig_keys)}] Downloading {rel_path} ...", end=" ")
    try:
        subprocess.run(
            ["aws", "s3", "cp", s3_path, local_path, "--no-sign-request"],
            capture_output=True, text=True, check=True,
        )
        print("OK")
    except subprocess.CalledProcessError as e:
        print(f"FAILED: {e.stderr.strip()}")
        failed.append(key)

print(f"\nDone. {len(blueconfig_keys) - len(failed)}/{len(blueconfig_keys)} downloaded successfully.")
if failed:
    print(f"{len(failed)} failed downloads:")
    for f in failed:
        print(f"  {f}")

## 4. Verify downloaded structure

In [ ]:
# Count downloaded BlueConfig files and show directory structure
downloaded = list(Path(OUTPUT_ROOT).rglob("BlueConfig"))
print(f"Downloaded {len(downloaded)} BlueConfig files to {OUTPUT_ROOT}\n")

# Show structure grouped by top-level directory
from collections import defaultdict
by_uuid = defaultdict(list)
for p in sorted(downloaded):
    rel = p.relative_to(OUTPUT_ROOT)
    parts = rel.parts
    by_uuid[parts[0]].append(str(rel))

for uuid_dir, files in sorted(by_uuid.items()):
    print(f"{uuid_dir}/ ({len(files)} BlueConfig files)")
    for f in sorted(files):
        print(f"  {f}")